# Unit 4 Hands-On ②: REINFORCE — Pixelcopter

## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 라이브러리 임포트
5. 환경 탐색
6. 정책 네트워크 정의
7. REINFORCE 훈련 함수
8. 하이퍼파라미터 설정 및 훈련
9. 에이전트 평가
10. 훈련 영상 저장
11. Hugging Face Hub 업로드


---
## 1. 환경 설치


In [1]:
%%capture
!apt install -y python-opengl ffmpeg xvfb
!pip install pyvirtualdisplay

In [2]:
# Pixelcopter는 PLE(PyGame Learning Environment)를 통해 제공됨
!pip install pygame
!pip install git+https://github.com/ntasfi/PyGame-Learning-Environment.git
!pip install imageio imageio-ffmpeg huggingface_hub torch torchvision

  Cloning https://github.com/ntasfi/PyGame-Learning-Environment.git to /tmp/pip-req-build-da_63jf7
  Running command git clone --filter=blob:none --quiet https://github.com/ntasfi/PyGame-Learning-Environment.git /tmp/pip-req-build-da_63jf7
  Resolved https://github.com/ntasfi/PyGame-Learning-Environment.git to commit 3dbe79dc0c35559bb441b9359948aabf9bb3d331
  Preparing metadata (setup.py) ... done
  Created wheel for ple: filename=ple-0.0.1-py3-none-any.whl size=50769 sha256=7678306919054245c99329f03b201e4d04074ebd9c5062b1d55c6461acaf6856
  Stored in directory: /tmp/pip-ephem-wheel-cache-btkoqcnr/wheels/6d/3c/74/aa0f046a54330af388e34b880213857c59e03b701cdcd9c38f
Successfully built ple


---
## 2. Google Drive 마운트


In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/RL_Course/Unit4_Pixelcopter'  # ✏️ 원하는 경로로 변경
VIDEO_DIR  = f'{DRIVE_BASE}/training_videos'
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'영상: {VIDEO_DIR}')
print(f'모델: {MODEL_DIR}')


Mounted at /content/drive
영상: /content/drive/MyDrive/RL_Course/Unit4_Pixelcopter/training_videos
모델: /content/drive/MyDrive/RL_Course/Unit4_Pixelcopter


---
## 3. 가상 디스플레이 설정


In [4]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()


---
## 4. 라이브러리 임포트


In [5]:
import numpy as np
from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# PLE를 통해 Pixelcopter 환경 직접 사용
from ple import PLE
from ple.games.pixelcopter import Pixelcopter

import imageio
from huggingface_hub import HfApi, login
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from IPython.display import Video, display
from pathlib import Path
import datetime, json, tempfile


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
couldn't import doomish
Couldn't import doom


---
## 5. 환경 탐색

**관측 공간** (7차원):
헬리콥터 y위치/y속도, 다음 블록까지의 거리/높이, 블록 상단y/하단y, 현재 높이

**행동 공간** (2개): `0` 하강(대기) / `1` 상승(엔진 점화)


In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)


사용 디바이스: cpu


In [7]:
# PLE 환경 생성
# display_screen=False: 창을 띄우지 않고 백그라운드 실행
game       = Pixelcopter(width=48, height=48)
env        = PLE(game, fps=30, display_screen=False)
env.init()

game_eval  = Pixelcopter(width=48, height=48)
eval_env   = PLE(game_eval, fps=30, display_screen=False)
eval_env.init()

# 행동 목록: [None(하강), 119(상승(스페이스바))]
action_set = env.getActionSet()

# 상태 공간 크기 확인
env.reset_game()
state_sample = np.array(list(env.getGameState().values()), dtype=np.float32)
s_size = len(state_sample)   # 7
a_size = len(action_set)     # 2

env_id = 'Pixelcopter-PLE-v0'

print('상태 공간 크기:', s_size)
print('행동 공간 크기:', a_size)
print('상태 키:', list(env.getGameState().keys()))


상태 공간 크기: 7
행동 공간 크기: 2
상태 키: ['player_y', 'player_vel', 'player_dist_to_ceil', 'player_dist_to_floor', 'next_gate_dist_to_player', 'next_gate_block_top', 'next_gate_block_bottom']


---
## 6. 정책 네트워크

CartPole(2층)보다 레이어를 하나 추가한 **3층 MLP**를 사용합니다.  
더 복잡한 환경에서의 표현력을 높이기 위함입니다.

```
입력(7) → FC1(h) → FC2(h×2) → FC3(2) → Softmax
```


In [8]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size * 2)
        self.fc3 = nn.Linear(h_size * 2, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        """상태(numpy array)를 받아 행동 인덱스와 log_prob 반환"""
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs  = self.forward(state).cpu()
        m      = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)


---
## 7. REINFORCE 훈련 함수

**PLE API**:
```python
env.reset_game()                                    # 에피소드 초기화
state = np.array(list(env.getGameState().values())) # 상태 획득
reward = env.act(action_set[action_idx])            # 행동 실행
done   = env.game_over()                            # 종료 여부
```

**REINFORCE 업데이트 공식**:
$$\mathcal{L} = -\sum_t \log \pi_\theta(a_t|s_t) \cdot G_t$$


In [9]:
def get_state(env):
    """PLE 환경의 상태 딕셔너리를 numpy 배열로 변환"""
    return np.array(list(env.getGameState().values()), dtype=np.float32)


def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    scores_deque = deque(maxlen=100)
    scores = []

    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []

        env.reset_game()
        state = get_state(env)

        for t in range(max_t):
            # 정책으로 행동 인덱스 선택 (0 또는 1)
            action_idx, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)

            # action_set[idx]로 실제 PLE 행동값 전달
            reward = env.act(action_set[action_idx])
            rewards.append(reward)

            if env.game_over():
                break

            state = get_state(env)

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # 누적 할인 보상 G_t = r_t + γ·G_{t+1} 역방향 계산
        returns = deque(maxlen=max_t)
        for t in range(len(rewards))[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        # Return 정규화 (훈련 안정화)
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # 정책 손실 계산 및 역전파
        policy_loss = torch.cat(
            [-log_prob * G for log_prob, G in zip(saved_log_probs, returns)]
        ).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores


---
## 8. 하이퍼파라미터 설정 및 훈련

| 파라미터 | 값 |
|---|---|
| `h_size` | 64 |
| `n_training_episodes` | 50,000 |
| `gamma` | 0.99 |
| `lr` | 1e-4 |


In [10]:
pixelcopter_hyperparameters = {
    'h_size': 64,
    'n_training_episodes': 20000,
    'n_evaluation_episodes': 10,
    'max_t': 10000,
    'gamma': 0.99,
    'lr': 1e-4,
    'env_id': env_id,
    'state_space': s_size,
    'action_space': a_size,
}


In [11]:
pixelcopter_policy = Policy(
    pixelcopter_hyperparameters['state_space'],
    pixelcopter_hyperparameters['action_space'],
    pixelcopter_hyperparameters['h_size']
).to(device)

pixelcopter_optimizer = optim.Adam(
    pixelcopter_policy.parameters(),
    lr=pixelcopter_hyperparameters['lr']
)


In [ ]:
scores = reinforce(
    pixelcopter_policy,
    pixelcopter_optimizer,
    pixelcopter_hyperparameters['n_training_episodes'],
    pixelcopter_hyperparameters['max_t'],
    pixelcopter_hyperparameters['gamma'],
    print_every=1000,
)


Episode 1000	Average Score: 6.21
Episode 2000	Average Score: 9.18
Episode 3000	Average Score: 9.20
Episode 4000	Average Score: 12.05


In [ ]:
# 훈련 곡선 시각화
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scores, alpha=0.3, label='Episode Score')
if len(scores) >= 100:
    moving_avg = [np.mean(scores[max(0, i-99):i+1]) for i in range(len(scores))]
    ax.plot(moving_avg, label='100 Episode Moving Average', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Score')
ax.set_title('Pixelcopter REINFORCE Training Curve')
ax.legend()
plt.tight_layout()
plt.show()


---
## 9. 에이전트 평가


In [ ]:
def evaluate_agent(env, action_set, max_steps, n_eval_episodes, policy):
    episode_rewards = []

    for _ in range(n_eval_episodes):
        env.reset_game()
        state = get_state(env)
        total_reward = 0.0

        for _ in range(max_steps):
            action_idx, _ = policy.act(state)
            total_reward += env.act(action_set[action_idx])
            if env.game_over():
                break
            state = get_state(env)

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


mean_reward, std_reward = evaluate_agent(
    eval_env, action_set,
    pixelcopter_hyperparameters['max_t'],
    pixelcopter_hyperparameters['n_evaluation_episodes'],
    pixelcopter_policy
)
print(f'평균 보상: {mean_reward:.2f} ± {std_reward:.2f}')


---
## 10. 훈련 영상 저장


In [ ]:
def record_video(env, action_set, policy, out_directory, fps=30):
    images = []
    env.reset_game()
    state = get_state(env)
    images.append(env.getScreenRGB())

    while not env.game_over():
        action_idx, _ = policy.act(state)
        env.act(action_set[action_idx])
        state = get_state(env)
        images.append(env.getScreenRGB())

    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)


video_path = f'{VIDEO_DIR}/pixelcopter_trained.mp4'
record_video(eval_env, action_set, pixelcopter_policy, video_path)
print(f'✅ 영상 저장: {video_path}')

display(Video(video_path, embed=True, width=400))


---
## 11. Hugging Face Hub 업로드

1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
# ✏️ 본인의 HF 토큰 입력
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')


In [ ]:
def push_to_hub(repo_id, model, hyperparameters, env, action_set, video_fps=30):
    _, repo_name = repo_id.split('/')
    api = HfApi()
    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpdirname:
        local_dir = Path(tmpdirname)

        # 모델 저장
        torch.save(model, local_dir / 'model.pt')

        # 하이퍼파라미터 저장
        with open(local_dir / 'hyperparameters.json', 'w') as f:
            json.dump(hyperparameters, f)

        # 평가 및 결과 저장
        mean_reward, std_reward = evaluate_agent(
            env, action_set,
            hyperparameters['max_t'],
            hyperparameters['n_evaluation_episodes'],
            model
        )
        with open(local_dir / 'results.json', 'w') as f:
            json.dump({
                'env_id': hyperparameters['env_id'],
                'mean_reward': mean_reward,
                'n_evaluation_episodes': hyperparameters['n_evaluation_episodes'],
                'eval_datetime': datetime.datetime.now().isoformat(),
            }, f)

        # 모델 카드
        env_name = hyperparameters['env_id']
        metadata = {'tags': [env_name, 'reinforce', 'reinforcement-learning',
                             'custom-implementation', 'deep-rl-class']}
        eval_meta = metadata_eval_result(
            model_pretty_name=repo_name,
            task_pretty_name='reinforcement-learning',
            task_id='reinforcement-learning',
            metrics_pretty_name='mean_reward',
            metrics_id='mean_reward',
            metrics_value=f'{mean_reward:.2f} +/- {std_reward:.2f}',
            dataset_pretty_name=env_name,
            dataset_id=env_name,
        )
        metadata = {**metadata, **eval_meta}

        readme_path = local_dir / 'README.md'
        readme_path.write_text(
            f'# **Reinforce** Agent playing **{env_name}**\n'
            f'Unit 4 of the Deep Reinforcement Learning Course\n',
            encoding='utf-8'
        )
        metadata_save(readme_path, metadata)

        # 영상 생성 및 업로드
        record_video(env, action_set, model, local_dir / 'replay.mp4', fps=video_fps)
        api.upload_folder(repo_id=repo_id, folder_path=local_dir, path_in_repo='.')
        print(f'✅ 업로드 완료: {repo_url}')
        print(f'   평균 보상: {mean_reward:.2f} ± {std_reward:.2f}')


In [ ]:
repo_id = 'YOUR_HF_USERNAME/Reinforce-Pixelcopter-PLE-v0'  # ✏️ 본인 username으로 변경

push_to_hub(
    repo_id=repo_id,
    model=pixelcopter_policy,
    hyperparameters=pixelcopter_hyperparameters,
    env=eval_env,
    action_set=action_set,
    video_fps=30
)
